# 🎁 Бонус — JWT, Swagger та CORS `🚀 Middle+`

Три речі, які майже завжди є в реальному DRF-проєкті й про які люблять спитати на співбесіді.
Junior-у досить **розуміти ідею** кожної; деталі — для Middle+.

**До уроків:** [Урок 1](lektsiya-1-django-rest-framework-osnovy.ipynb), [Урок 2](lektsiya-2-viewsets-routery-autentyfikatsiya.ipynb).

## 1. JWT-автентифікація (`djangorestframework-simplejwt`)

**Token Auth** з Уроку 2 зберігає токени в БД (при кожному запиті — похід у базу). **JWT**
(JSON Web Token) — це **самодостатній підписаний токен**: сервер перевіряє підпис, не звертаючись
до БД. Токен складається з даних (payload) + підпису; має **строк дії**.

```bash
pip install djangorestframework-simplejwt
```
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": [
        "rest_framework_simplejwt.authentication.JWTAuthentication",
    ],
}
```
```python
# myproject/urls.py
from rest_framework_simplejwt.views import TokenObtainPairView, TokenRefreshView
from django.urls import path

urlpatterns = [
    path("api/token/", TokenObtainPairView.as_view()),      # логін -> access + refresh
    path("api/token/refresh/", TokenRefreshView.as_view()), # оновити access за refresh
]
```
```bash
# 1. Логін -> два токени
curl -X POST http://127.0.0.1:8000/api/token/ -d "username=ivan&password=secret"
# -> {"refresh": "eyJhbG...", "access": "eyJhbG..."}

# 2. Запит із access-токеном
curl http://127.0.0.1:8000/api/books/ -H "Authorization: Bearer eyJhbG..."
```

- **access** — короткоживучий токен (напр. 5 хв), яким ходять у кожен запит;
- **refresh** — довгоживучий (напр. дні), яким отримують новий `access`, не логінячись знову.

> 🎯 **Питання: чим JWT кращий за звичайний Token?** JWT **stateless** — сервер не тримає токени в
> БД, тому легше масштабується на багато серверів. Токен має строк дії (безпечніше). Мінус:
> достроково «відкликати» JWT складніше (для цього ведуть чорні списки refresh-токенів).
> `Authorization: Bearer <jwt>` (а не `Token <...>`).

## 2. Swagger / OpenAPI — авто-документація API (`drf-spectacular`)

Щоб фронтенд/тестувальники бачили всі ендпоінти, поля й коди відповідей, API документують за
стандартом **OpenAPI** і показують через **Swagger UI** — інтерактивну сторінку, де запити можна
пробувати прямо в браузері. `drf-spectacular` генерує це **автоматично** з ваших serializers/views.

```bash
pip install drf-spectacular
```
```python
# settings.py
INSTALLED_APPS += ["drf_spectacular"]
REST_FRAMEWORK = {
    "DEFAULT_SCHEMA_CLASS": "drf_spectacular.openapi.AutoSchema",
}
```
```python
# myproject/urls.py
from drf_spectacular.views import SpectacularAPIView, SpectacularSwaggerView
from django.urls import path

urlpatterns = [
    path("api/schema/", SpectacularAPIView.as_view()),                       # OpenAPI-схема (JSON/YAML)
    path("api/docs/", SpectacularSwaggerView.as_view(url_name="schema")),    # Swagger UI
]
```
Відкрийте `http://127.0.0.1:8000/api/docs/` — отримаєте живу документацію всього API.

> 🔎 **Навіщо це:** документація не «розходиться» з кодом (генерується з нього), а фронтенд може
> почати роботу, не питаючи бекендера про кожне поле. Альтернатива — `drf-yasg` (робить те саме).

## 3. CORS — доступ із фронтенду на іншому домені (`django-cors-headers`)

Браузер за замовчуванням **блокує** запити зі сторінки одного походження (домен+порт) до API на
**іншому** (**Same-Origin Policy**). Тобто фронтенд на `http://localhost:3000` не зможе звертатись
до API на `http://localhost:8000` — поки бекенд явно це **не дозволить** через **CORS**
(Cross-Origin Resource Sharing).

```bash
pip install django-cors-headers
```
```python
# settings.py
INSTALLED_APPS += ["corsheaders"]

MIDDLEWARE = [
    "corsheaders.middleware.CorsMiddleware",     # ← якомога вище у списку
    "django.middleware.common.CommonMiddleware",
    # ... решта middleware
]

# Дозволені джерела (домени фронтенду):
CORS_ALLOWED_ORIGINS = [
    "http://localhost:3000",
    "https://myfrontend.com",
]
```

> 🎯 **Питання: що таке CORS і навіщо?** Це механізм, яким **сервер** дозволяє браузеру приймати
> відповіді на крос-доменні запити. Він захищає користувача (Same-Origin Policy), а `CORS_ALLOWED_ORIGINS`
> — це «білий список» доменів, яким ваше API довіряє. ⚠️ Не ставте `CORS_ALLOW_ALL_ORIGINS = True`
> у продакшені — це відкриває API всім.

# ✅ Підсумок бонусу
- **JWT** (`simplejwt`) — stateless підписаний токен зі строком дії: `access` (короткий) + `refresh` (довгий); заголовок `Authorization: Bearer <jwt>`.
- **Swagger/OpenAPI** (`drf-spectacular`) — авто-документація API, яку можна пробувати в браузері (`/api/docs/`).
- **CORS** (`django-cors-headers`) — дозвіл браузеру ходити на API з **іншого** домену; білий список `CORS_ALLOWED_ORIGINS`.

### 📚 Хочу знати більше
- SimpleJWT: <https://django-rest-framework-simplejwt.readthedocs.io/>
- drf-spectacular: <https://drf-spectacular.readthedocs.io/>
- django-cors-headers: <https://github.com/adamchainz/django-cors-headers>
- Що таке CORS (MDN): <https://developer.mozilla.org/en-US/docs/Web/HTTP/CORS>